In [2]:
import open3d as o3d
import numpy as np
import matplotlib.pyplot as plt #matplotlib绘图库
import copy #python的copy模块提供了浅拷贝和深拷贝操作
import os
import sys

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


In [3]:
#读取并渲染点云图
print("Load a ply point cloud, print it, and render it")
pcd = o3d.io.read_point_cloud(r"C:\Users\Jin Cai\Desktop\open3D_tutorial\data\extract\DemoCropPointCloud\fragment.ply")
print(pcd)
print(np.asarray(pcd.points))
o3d.visualization.draw_geometries([pcd])

Load a ply point cloud, print it, and render it
PointCloud with 196133 points.
[[0.65234375 0.84686458 2.37890625]
 [0.65234375 0.83984375 2.38430572]
 [0.66737998 0.83984375 2.37890625]
 ...
 [2.00839925 2.39453125 1.88671875]
 [2.00390625 2.39488506 1.88671875]
 [2.00390625 2.39453125 1.88793314]]


### DBSCAN Clustering 距离+密度
Open3D implements DBSCAN [Ester1996] that is a density based clustering algorithm. The algorithm is implemented in `cluster_dbscan` and requires two parameters: `eps` defines the distance to neighbors in a cluster and `min_points` defines the minimum number of points required to form a cluster. The function returns `labels`, where the label `-1` indicates noise.

Open3D 实现了 DBSCAN [Ester1996]，这是一种基于密度的聚类算法。
该算法通过 `cluster_dbscan` 实现，需要两个参数：
- `eps`定义一个簇中点与其邻居之间允许的距离范围。
- `min_points`定义形成一个簇所需要的最少点数。  
该函数返回 `labels`（标签），其中标签 `-1` 表示噪声点

**离得近+比较密，同时把零散的点识别为噪音**

In [4]:
#DBSCAN聚类 -> 给每个点一个类别编号 -> 给不同的编号上色 -> 显示点云

#临时把 Open3D 的日志输出级别设置成 Debug，让你看到 DBSCAN 运行过程中的详细信息，这两行可以不写
with o3d.utility.VerbosityContextManager(
        o3d.utility.VerbosityLevel.Debug) as cm:
    #DBSCAN会给每个点分配一个标签，如0，1，2，3，-1等，其中-1表示噪声点，其他的表示cluster0,1,2,3等
    #如labels = [0, 0, 0, 1, 1, 1, 1, -1, -1, 2]
    labels = np.array(
        #执行DBSCAN
        pcd.cluster_dbscan(
            eps=0.02,  #邻居的最大半径，即以点p为中心，半径为eps的球体内的点被认为是邻居
            min_points=10,  #在这个球内至少有10个邻居才被认为是够密
            print_progress=True)
        )
#一共分了多少个cluster，max_label表示最大的cluster编号，cluster编号是从0开始的
#如labels = [0, 0, 1, 1, 2, 2, -1]，那么max_label = 2，表示有3个cluster，编号为0,1,2
max_label = labels.max()
print(f"point cloud has {max_label + 1} clusters")
#用matplotlib给不同的cluster上色
colors = plt.get_cmap("tab20")(
    labels / (max_label if max_label > 0 else 1)
)
#噪声变成黑色
colors[labels < 0] = 0
#把颜色赋给点云，colors[:, :3] 的意思就是只取 RGB 三个通道
pcd.colors = o3d.utility.Vector3dVector(colors[:, :3])
o3d.visualization.draw_geometries([pcd])

[Open3D DEBUG] Precompute neighbors.
[Open3D DEBUG] Done Precompute neighbors.
[Open3D DEBUG] Compute Clusters
[Open3D DEBUG] Done Compute Clusters: 10
point cloud has 10 clusters
